<a href="https://colab.research.google.com/github/Abdoulaye279/Quantum-Computing/blob/main/05_qiskit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Export to Qiskit

The simulated gates as Qiskit circuits, and a depolarizing noise model tuned to the same Bell fidelity as the full error model of 02. Qubit order: ion 1 = q1, ion 2 = q0.

In [1]:
# colab: get the repo first
try:
    import google.colab
    !git clone -q https://github.com/Abdoulaye279/Quantum-Computing
    %cd Quantum-Computing
except ImportError:
    pass

In [2]:
!pip install -q qiskit qiskit-aer

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from cirac_zoller import *

labels = ["SS", "SD", "DS", "DD"]
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, process_fidelity
from qiskit.circuit.library import RXXGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
plt.style.use("quant.mplstyle")

In [4]:
cz_circ = QuantumCircuit(2)
cz_circ.sdg(0); cz_circ.cx(1, 0); cz_circ.s(0)   # CY, ion 1 control
Ue = eff_unitary(gate_amp(0.0))
print("simulated CZ sequence vs CY circuit:", round(process_fidelity(Operator(Ue), Operator(cz_circ)), 6))
ref = QuantumCircuit(2); ref.cx(1, 0)
print("simulated CZ sequence vs plain CNOT:", round(process_fidelity(Operator(Ue), Operator(ref)), 3))
# 1.0 with CY, 0.25 with a plain CNOT: the local S rotations matter once the gate is inside a circuit

simulated CZ sequence vs CY circuit: 1.0
simulated CZ sequence vs plain CNOT: 0.25


In [5]:
ms_circ = QuantumCircuit(2); ms_circ.append(RXXGate(np.pi / 2), [0, 1])
print("simulated MS vs RXX(pi/2):", round(process_fidelity(Operator(eff_unitary(ms_gate(2 * np.pi / 300 / 4))), Operator(ms_circ)), 6))
cnot_ms = QuantumCircuit(2)
cnot_ms.ry(np.pi / 2, 1); cnot_ms.append(RXXGate(np.pi / 2), [1, 0]); cnot_ms.rx(-np.pi / 2, 1); cnot_ms.rx(-np.pi / 2, 0); cnot_ms.ry(-np.pi / 2, 1)
print("CNOT built from MS vs CX:", round(process_fidelity(Operator(cnot_ms), Operator(ref)), 6))
print(cnot_ms.draw())
# MS = RXX(pi/2), and one MS + single-qubit rotations gives a CNOT (Maslov 2017)
# the "not TP" warning comes from the truncated phonon space: a tiny bit of population leaks out of n=0

Input channel is not TP. Tr_2[Choi] - I has non-zero eigenvalues: [-5.19734613e-06 -5.19734613e-06]


simulated MS vs RXX(pi/2): 0.999997
CNOT built from MS vs CX: 1.0
                ┌───────────┐┌──────────┐            
q_0: ───────────┤1          ├┤ Rx(-π/2) ├────────────
     ┌─────────┐│  Rxx(π/2) │├──────────┤┌──────────┐
q_1: ┤ Ry(π/2) ├┤0          ├┤ Rx(-π/2) ├┤ Ry(-π/2) ├
     └─────────┘└───────────┘└──────────┘└──────────┘


In [6]:
rn = run(bell_input, shots=300, nth=0.02, eps=0.05, amp_sig=0.03, ph_sig=0.32)   # all errors, as in 02
F = bell_fidelity(rn)
lam = (1 - F) * 4 / 3            # depolarizing: F = 1 - 3 lam / 4
noise = NoiseModel(); noise.add_all_qubit_quantum_error(depolarizing_error(lam, 2), ["cx"])
sim = AerSimulator(noise_model=noise)
bell = QuantumCircuit(2, 2); bell.h(1); bell.cx(1, 0); bell.measure([1, 0], [0, 1])
counts = sim.run(transpile(bell, sim), shots=20000, seed_simulator=1).result().get_counts()
print(f"depolarizing lambda = {lam:.3f}:", {k: round(v / 20000, 3) for k, v in sorted(counts.items())})
print("paper: P(SS) = 0.42, P(DD) = 0.45")
print("physical model:     ", dict(zip(labels, np.round(np.real(np.diag(rn)), 3))))
# same fidelity by construction, but depolarizing keeps SS and DD equal (0.41 / 0.41)
# the physical model doesn't (0.37 / 0.46), and the paper has DD above SS too (0.42 / 0.45) -> right number, wrong shape

depolarizing lambda = 0.355: {'00': 0.405, '01': 0.092, '10': 0.095, '11': 0.409}
paper: P(SS) = 0.42, P(DD) = 0.45
physical model:      {'SS': np.float64(0.372), 'SD': np.float64(0.081), 'DS': np.float64(0.092), 'DD': np.float64(0.455)}
